# TFT experiment - Walmart Store Sales Forecasting

Person B track (DL + Prophet). Third of the four notebooks in this track,
after `model_experiment_dlinear.ipynb` and `model_experiment_nbeats.ipynb`.

**Library switch (this revision):** previously built on `darts.models.TFTModel`,
which scored Kaggle Public 3615.58 / Private 3779.46 - worse than every
non-DL model in the project. Course feedback on the presentation was that
Darts was the wrong tool for TFT/N-BEATS and a neural-network-native
forecasting library should have been used instead. This notebook is rebuilt
on **Nixtla's `neuralforecast`**, which implements TFT directly rather than
wrapping it. This also removes a real, previously-documented bug class: Darts'
`Scaler` matched fitted per-series transformers too series by *list position*,
which silently corrupted predictions for both N-BEATS and TFT (see
`README.md` SS4.5/4.6). `neuralforecast` keys everything by an explicit
`unique_id` column, so that bug class doesn't exist here, and per-series
target scaling is handled internally by the model (`scaler_type=...`) rather
than by hand.

**Decisions this notebook records for the README:**
- future vs past covariate split - reused as-is from the DLinear/N-BEATS EDA
  finding (checked once against `features.csv` coverage, not re-derived here).
- static covariate scope - `Type` (one-hot) + `Size` (scaled) only, **not**
  raw Store/Dept identity embeddings. `neuralforecast`'s static covariates
  (`stat_exog_list`) are plain numeric columns, not a Darts-style
  `categorical_embedding_sizes` table, so one-hotting ~144 Store/Dept
  categories would be a heavy, untested scope increase for little expected
  gain - the old Darts run's own diagnostics (README SS4.6) already showed
  `Size` dominated static-covariate importance.
- quantile loss (`MQLoss(quantiles=[0.1, 0.5, 0.9])`) - the median is used as
  the point forecast for WMAE/submission, the 10-90% band for interval plots.
- **lag-52 seasonal-naive blend (this revision's score lever).** A first
  neuralforecast pass scored Kaggle Public 3566 - better than the old Darts
  TFT (3615), but its **CV WMAE was ~1531** (better than Prophet's CV) while
  Kaggle was 3566, a 2.3x gap. That gap is *not* undertraining (the model fits
  CV well): it's the holiday regime. The 13-week CV folds never contain
  Thanksgiving/Christmas, but the real 39-week test contains both, weighted 5x
  by WMAE, and TFT **smooths holiday amplitude** (README SS4.6 - peaks too low).
  The fix is post-processing: blend the TFT median with the **lag-52
  seasonal-naive** baseline (same-week-last-year, which carries the full
  holiday peak since seasonality >> trend here, README SS4.1/4.3):
  `submission = w*TFT + (1-w)*seasonal`, clipped >= 0. **This means the
  submission is TFT blended with a seasonal-naive baseline, not pure TFT** -
  reported honestly as such.
- **blend weight tuned on a TESTLIKE holdout, not the 13-week CV.** The CV
  can't tune a holiday knob, so - exactly as `README.md` describes for Prophet
  - a **TESTLIKE** fold copies the real test structure (train through
  2011-10-28, forecast the 39 weeks 2011-11 -> 2012-07 including Thanksgiving +
  Christmas 2011) and the blend weight `w` is swept on it by re-scoring (no
  retrain per weight). Its WMAE is the honest, Kaggle-predictive number; the
  13-week CV is now a **single fold**, kept only as a quick sanity check.
- `h` (forecast horizon) is fixed at model-construction time in
  `neuralforecast` - unlike Darts, a single fitted model can't serve both a
  13-week CV-fold forecast and a 39-week submission forecast. HP-search/CV
  use `h=VAL_WEEKS` (13); the TESTLIKE and Final stages use `h=HORIZON` (39).
- **compute:** hard `max_time` wall-clock ceilings per stage (ported from the
  nbeats notebook) plus a visible percent-progress callback, so the whole run
  stays comfortably under 2 hours and never becomes a silent black box.
- **dropped:** `TFTExplainer`-style attention/variable-selection plots -
  Darts-specific, no `neuralforecast` equivalent. Not replaced with a
  substitute.

**MLflow run plan** (under `TFT_Training`, same experiment as the Darts runs
so history stays comparable - each run additionally logs
`library="neuralforecast"` to distinguish them): `TFT_Feature_Selection`,
three `TFT_HP_*` architecture-comparison runs, one `TFT_CV_Fold0` sanity fold,
`TFT_TestLike` (the honest holiday-containing holdout + blend-weight sweep),
and `TFT_Final`.


##  Init cell (Colab-compatible)

Byte-identical to the DLinear/N-BEATS notebooks' init cell, plus a
`neuralforecast` install alongside `darts[torch]` (DLinear still needs Darts;
only TFT is switching libraries in this revision).


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IS_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if IS_COLAB:
    REPO_URL = "https://github.com/NikaMikeltadze/walmart-sales-forecasting.git"
    REPO_DIR = "/content/walmart-sales-forecasting"

    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r",
         f"{REPO_DIR}/requirements.txt", "--quiet"],
        check=True,
    )
    subprocess.run([sys.executable, "-m", "pip", "install", "neuralforecast", "--quiet"], check=True)

    os.chdir(f"{REPO_DIR}/notebooks")

    from google.colab import drive
    drive.mount("/content/drive")

    drive_data_dir = Path("/content/drive/MyDrive/walmart-sales-forecasting/data/raw")
    repo_data_dir = Path(REPO_DIR) / "data" / "raw"
    if drive_data_dir.exists():
        subprocess.run(["cp", "-r", f"{drive_data_dir}/.", str(repo_data_dir)], check=True)
    else:
        raise FileNotFoundError(
            f"Expected Drive data folder not found at {drive_data_dir}. "
            "Create it (or add it as a My Drive shortcut) before running this notebook."
        )

sys.path.append(str(Path.cwd().parent))


##  Imports

In [ ]:
import pickle
import tempfile
import time
from datetime import timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow

from sklearn.preprocessing import StandardScaler

import torch
from neuralforecast import NeuralForecast
from neuralforecast.models import TFT
from neuralforecast.losses.pytorch import MQLoss

from pytorch_lightning.loggers import CSVLogger
from pytorch_lightning.callbacks import Callback

from src.preprocessing import load_raw_data, WalmartPreprocessor
from src.features import add_temporal_features
from src.evaluation import weighted_mae

pd.set_option("display.max_columns", 50)
pd.set_option("future.no_silent_downcasting", True)


###  Manual credentials override (VS Code / non-Colab-UI runtimes)

`google.colab.userdata` (Colab Secrets) can only be read from the Colab
**browser UI**. When the Colab runtime is driven from VS Code or any other
non-UI frontend it times out (`Secrets can only be fetched when running from
the Colab UI`). This cell sets the DagsHub creds directly instead, and the
credentials cell below skips `userdata` whenever these env vars are already
set.

`getpass` is used so the token is never written into this committed notebook -
run the cell and paste the values when prompted. Leave a prompt blank to fall
through to Colab Secrets / `.env` below (e.g. when you *are* on the Colab UI).


In [ ]:
import os
from getpass import getpass

# Only prompt for values not already set, so re-running cells doesn't re-ask.
# Leave a prompt blank to fall through to Colab Secrets / .env in the next cell.
if not os.environ.get("MLFLOW_TRACKING_USERNAME"):
    _user = getpass("DagsHub username (blank -> use Colab Secrets / .env): ").strip()
    if _user:
        os.environ["MLFLOW_TRACKING_USERNAME"] = _user
if not os.environ.get("MLFLOW_TRACKING_PASSWORD"):
    _token = getpass("DagsHub token (blank -> use Colab Secrets / .env): ").strip()
    if _token:
        os.environ["MLFLOW_TRACKING_PASSWORD"] = _token


##  Load DagsHub credentials

`MLFLOW_TRACKING_USERNAME`/`MLFLOW_TRACKING_PASSWORD` are never hardcoded in
this notebook (it gets committed to the shared repo, so a hardcoded token
would leak).

- On the Colab UI: read from Colab secrets - add `DAGSHUB_USERNAME` and
  `DAGSHUB_TOKEN` via the key icon in the left sidebar, and enable
  "Notebook access" for both. Same approach as every other notebook.
- From VS Code / non-UI runtimes: use the manual-override cell above.
- Locally: falls back to a gitignored `.env` in the repo root.


In [ ]:
if os.environ.get("MLFLOW_TRACKING_USERNAME") and os.environ.get("MLFLOW_TRACKING_PASSWORD"):
    # Already provided (e.g. by the manual-override cell above when driving the
    # Colab runtime from VS Code, where google.colab.userdata would time out).
    # Note: userdata.get(...) must NOT be evaluated in this case - it blocks for
    # ~minutes and raises when there is no Colab browser UI to answer it.
    creds_source = "pre-set environment variables"
elif IS_COLAB:
    from google.colab import userdata

    os.environ["MLFLOW_TRACKING_USERNAME"] = userdata.get("DAGSHUB_USERNAME")
    os.environ["MLFLOW_TRACKING_PASSWORD"] = userdata.get("DAGSHUB_TOKEN")
    creds_source = "Colab secrets (DAGSHUB_USERNAME / DAGSHUB_TOKEN)"
else:
    from dotenv import load_dotenv

    env_path = Path.cwd().parent / ".env"
    load_dotenv(env_path)
    creds_source = str(env_path)

assert os.environ.get("MLFLOW_TRACKING_USERNAME") and os.environ.get("MLFLOW_TRACKING_PASSWORD"), (
    f"MLFLOW_TRACKING_USERNAME/PASSWORD not set (tried: {creds_source}). "
    "On the Colab UI: add DAGSHUB_USERNAME and DAGSHUB_TOKEN as Colab secrets "
    "(key icon in the left sidebar) and enable notebook access for both. "
    "From VS Code / non-UI runtimes: run the manual-override cell above. "
    "Locally: create a .env in the repo root with MLFLOW_TRACKING_USERNAME=... "
    "and MLFLOW_TRACKING_PASSWORD=..."
)
print("MLflow credentials loaded from:", creds_source)


##  MLflow tracking store

Shared DagsHub MLflow server - the single source of truth for cross-model
WMAE comparison and the Model Registry, so all models log here rather than to
a per-notebook local store. Same experiment name as the previous Darts-based
runs (`TFT_Training`) so history stays comparable; each run below logs
`library="neuralforecast"` as a param to distinguish new runs from old ones.
Does not silently fall back to a local store if auth fails - that would
desync TFT's runs from the rest of the team's.


In [ ]:
MLFLOW_TRACKING_URI = "https://dagshub.com/NikaMikeltadze/walmart-sales-forecasting.mlflow"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

try:
    mlflow.set_experiment("TFT_Training")
    mlflow.MlflowClient().search_experiments(max_results=1)  # force a network round trip now
except Exception as e:
    raise RuntimeError(
        "Could not authenticate to the DagsHub MLflow server at "
        f"{MLFLOW_TRACKING_URI}. Set MLFLOW_TRACKING_USERNAME and "
        "MLFLOW_TRACKING_PASSWORD (a DagsHub access token) as environment "
        "variables, then re-run this cell. Not falling back to local "
        "./mlruns or sqlite - that would desync from the rest of the team's runs."
    ) from e

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Active experiment:", mlflow.get_experiment_by_name("TFT_Training").name)


##  Load, merge, clean

Reuses `load_raw_data` / `WalmartPreprocessor` rather than reimplementing
loading logic. `WalmartPreprocessor.fit()` loads and caches `features_` /
`stores_` internally - `stores_` (Store, Type, Size) is what the
static-covariate section below builds on.


In [ ]:
raw_data = load_raw_data(data_dir="../data/raw")

train_raw = raw_data["train"]
test_raw = raw_data["test"]

preprocessor = WalmartPreprocessor(data_dir="../data/raw")
preprocessor.fit(train_raw)

train_clean = preprocessor.transform(train_raw)
test_clean = preprocessor.transform(test_raw)

train_feat = add_temporal_features(train_clean)
test_feat = add_temporal_features(test_clean)

print(train_feat.shape, test_feat.shape)
train_feat.head()


##  Covariate coverage decision

Same check already established in the DLinear/N-BEATS/old-TFT notebooks -
reused here rather than re-derived, since it's a property of `features.csv`,
not of the model. A column only becomes a *future* covariate if it's
non-null for every week of the test horizon (2012-11-02 to 2013-07-26).


In [ ]:
test_dates = pd.date_range("2012-11-02", "2013-07-26", freq="W-FRI")

macro_cols = ["Temperature", "Fuel_Price", "CPI", "Unemployment",
              "MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5"]

features_lookup = preprocessor.features_

coverage = {}
for col in macro_cols:
    non_null_dates = set(features_lookup.loc[features_lookup[col].notna(), "Date"])
    coverage[col] = set(test_dates).issubset(non_null_dates)

coverage_df = pd.Series(coverage, name="covers_full_test_range")
print(coverage_df)

calendar_future_cols = ["IsHoliday", "week_of_year", "month", "year",
                         "days_to_super_bowl", "days_to_labor_day",
                         "days_to_thanksgiving", "days_to_christmas"]

FUTURE_COVARIATE_COLS = calendar_future_cols + [c for c in coverage_df.index if coverage_df[c]]
PAST_COVARIATE_COLS = [c for c in coverage_df.index if not coverage_df[c]]
all_covariate_cols = FUTURE_COVARIATE_COLS + PAST_COVARIATE_COLS

print("Future covariates:", FUTURE_COVARIATE_COLS)
print("Past covariates:", PAST_COVARIATE_COLS)


##  Static covariates (`Type` + `Size` only - see intro markdown for why)

`neuralforecast`'s `stat_exog_list` takes plain numeric columns (no
Darts-style `categorical_embedding_sizes` table), so `Type` is one-hot
encoded and `Size` is scaled with `StandardScaler` (fit on the store table,
which has no train/test split of its own). One static row is built per
`unique_id` (`"{Store}_{Dept}"`), covering every (Store, Dept) pair seen in
train **or** test, so the pipeline's `.predict()` never hits a missing
static-covariate lookup.


In [ ]:
stores_df = preprocessor.stores_.copy()

type_dummies = pd.get_dummies(stores_df["Type"].astype(str), prefix="Type").astype(float)
TYPE_COLS = sorted(type_dummies.columns.tolist())

size_scaler = StandardScaler()
stores_df["Size_Scaled"] = size_scaler.fit_transform(stores_df[["Size"]]).astype(float)

stores_static = pd.concat([stores_df[["Store", "Size_Scaled"]], type_dummies], axis=1)
STATIC_COLS = TYPE_COLS + ["Size_Scaled"]

all_pairs = (
    pd.concat([train_feat[["Store", "Dept"]], test_feat[["Store", "Dept"]]])
    .drop_duplicates()
    .reset_index(drop=True)
)

static_df = all_pairs.merge(stores_static, on="Store", how="left")
static_df["unique_id"] = static_df["Store"].astype(str) + "_" + static_df["Dept"].astype(str)
static_df = static_df[["unique_id"] + STATIC_COLS]

print("Static covariates:", STATIC_COLS)
print(f"static_df: {static_df.shape[0]} (Store, Dept) series")
static_df.head()


##  Model constants and compute budget

**Runtime budget (kept under 2 hours):** `neuralforecast` trains ALL series
inside one global `nf.fit()` call per model, so the step budgets below are
**total gradient steps across the whole ~3,300-series panel**. They were
trimmed from an earlier 250/1000/4000 pass once a real Colab run showed the
model already fits well by ~1000 steps (CV WMAE ~1531) - the extra steps
weren't buying accuracy, they were just burning the time budget. The freed
time instead funds the new **TESTLIKE** holdout (the honest, holiday-containing
validation) and the **1-fold** CV (down from 3). Each stage also has a hard
`max_time` wall-clock ceiling (`*_MAX_TIME_MINUTES`, enforced via
`pl.Trainer(max_time=...)`), so whichever of steps-or-time is hit first ends
that fit - a real safety net after a prior run went 150+ minutes.
`start_padding_enabled=True` zero-pads any series shorter than
`input_size + h`, so none is dropped for short history.

`h` (forecast horizon) is fixed per model at construction time - HP-search and
the 1-fold CV use `h=VAL_WEEKS` (13); the TESTLIKE and Final stages use
`h=HORIZON` (39, the full test horizon).

**GPU check is a hard failure, not just a print.** Selecting "T4 GPU" in
Colab's Runtime > Change runtime type does NOT attach a GPU to a session
that's already running - a previous run silently trained on CPU because of
exactly this and ran 150+ minutes with nothing to show for it. If on Colab
without a GPU actually attached, the cell below raises immediately with
reconnect instructions. **Progress is visible**: `enable_progress_bar=True`
plus a custom `PercentProgressCallback` (prints
`step X/Y - Z% (bound by step/time budget) - elapsed Ns, ~W min remaining`
every 10% of whichever limit is closer) on every fit - never a silent black box.


In [ ]:
FREQ = "W-FRI"
inferred_freq = pd.infer_freq(sorted(train_feat["Date"].unique()))
print("Inferred frequency from data:", inferred_freq)
assert inferred_freq in ("W-FRI", None), f"Unexpected frequency: {inferred_freq}"

INPUT_SIZE = 52   # one year of history
HORIZON = 39      # covers the full test horizon - test is ~39 weeks past train
VAL_WEEKS = 13    # CV fold validation window - same convention as every other model

QUANTILES = [0.1, 0.5, 0.9]

GPU_AVAILABLE = torch.cuda.is_available()
print("=" * 60)
print("GPU available:", GPU_AVAILABLE)
print("=" * 60)

if IS_COLAB and not GPU_AVAILABLE:
    raise RuntimeError(
        "Running on Colab but no GPU is attached to THIS runtime session - "
        "training would silently fall back to CPU, and what should take "
        "~20-40 min could instead take hours with nothing to show for it "
        "(this already happened once: a 4000-step final fit ran 150+ minutes "
        "on CPU because of exactly this). Selecting 'T4 GPU' in Runtime > "
        "Change runtime type does NOT attach a GPU to a session that's "
        "already running - go to Runtime > Disconnect and delete runtime, "
        "then reconnect (or Runtime > Run all from a fresh state) so it "
        "actually starts on a GPU instance. Re-run this cell after "
        "reconnecting and confirm it prints 'GPU available: True' before "
        "continuing."
    )

# Step budgets - see markdown above for why these are NOT per-series like Darts' n_epochs.
# Trimmed from an earlier 250/1000/4000 pass: the user's real Colab log showed the model
# already fits CV well at 1000 steps (CV WMAE ~1531), so 4000 for the final was overkill -
# and the freed time goes to the new TESTLIKE holdout below. CV is also cut from 3 folds
# to 1 (see N_SPLITS) since the 13-week CV can't tune the holiday knob anyway.
HP_MAX_STEPS = 200         # ceiling for the quick 3-way architecture comparison
CV_MAX_STEPS = 800         # ceiling per CV fold (sanity check only now)
TESTLIKE_MAX_STEPS = 1200  # ceiling for the one holiday-containing 39-week holdout fit
FINAL_MAX_STEPS = 1500     # ceiling for the single final fit on full history

# Hard wall-clock ceilings (enforced via pl.Trainer's max_time), ported from the nbeats
# notebook - belt-and-suspenders against another runaway run (a prior run hit 150+ min).
# Worst case: 3*HP + 1*CV + 1*TESTLIKE + 1*FINAL = 3*4 + 15 + 25 + 30 = 82 min of training,
# leaving comfortable headroom under a 2-hour budget for data prep/diagnostics/submission.
HP_MAX_TIME_MINUTES = 4
CV_MAX_TIME_MINUTES = 15
TESTLIKE_MAX_TIME_MINUTES = 25
FINAL_MAX_TIME_MINUTES = 30

BATCH_SIZE_SEARCH = 512  # HP search + CV folds
BATCH_SIZE_FINAL = 1024


class PercentProgressCallback(Callback):
    """Prints plain-text '[TFT] step X/Y - Z% (bound by step/time budget) - elapsed
    Ns, ~W min remaining' progress at regular intervals - a guaranteed-visible
    supplement to the trainer's own progress bar, since that bar can render oddly or
    get lost in scrolled/captured notebook output. A previous run appeared to hang
    for 150+ minutes with zero visible feedback - this (plus max_time) ensures a long
    fit is never a silent black box again. Tracks progress against BOTH the step
    ceiling and the max_time ceiling (whichever is closer), since either can end it.
    """

    def __init__(self, max_time_seconds=None, print_every_pct=10):
        self.max_time_seconds = max_time_seconds
        self.print_every_pct = print_every_pct
        self._last_pct = -1
        self._t0 = None

    def on_train_start(self, trainer, pl_module):
        self._t0 = time.time()
        self._last_pct = -1

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        max_steps = trainer.max_steps
        elapsed = time.time() - self._t0
        step = trainer.global_step

        step_pct = (100 * step / max_steps) if max_steps and max_steps > 0 else 0
        time_pct = (100 * elapsed / self.max_time_seconds) if self.max_time_seconds else 0
        pct = int(max(step_pct, time_pct))

        if pct >= self._last_pct + self.print_every_pct:
            self._last_pct = pct
            rate = step / elapsed if elapsed > 0 else 0
            steps_remaining_min = (max_steps - step) / rate / 60 if (rate > 0 and max_steps) else float("inf")
            time_remaining_min = (self.max_time_seconds - elapsed) / 60 if self.max_time_seconds else float("inf")
            remaining_min = min(steps_remaining_min, time_remaining_min)
            bound_by = "step budget" if steps_remaining_min <= time_remaining_min else "time budget"
            print(f"[TFT] step {step}/{max_steps} - {pct}% (bound by {bound_by}) - elapsed {elapsed:.0f}s, ~{remaining_min:.1f} min remaining")


def make_trainer_kwargs(logger=None, max_time_minutes=None):
    """Keyword args forwarded through TFT's **trainer_kwargs to the underlying
    pl.Trainer, so GPU selection and the wall-clock time budget are consistent
    everywhere below."""
    max_time_seconds = max_time_minutes * 60 if max_time_minutes is not None else None
    kwargs = {
        "accelerator": "gpu" if GPU_AVAILABLE else "cpu",
        "enable_progress_bar": True,
        "callbacks": [PercentProgressCallback(max_time_seconds=max_time_seconds)],
    }
    if logger is not None:
        kwargs["logger"] = logger
    if GPU_AVAILABLE:
        kwargs["devices"] = 1
    if max_time_minutes is not None:
        kwargs["max_time"] = timedelta(minutes=max_time_minutes)
    return kwargs


def median_column(preds: pd.DataFrame) -> str:
    """Name of the median-quantile column in a neuralforecast predict() output -
    looked up by suffix rather than hardcoded, since it depends on the model's alias."""
    candidates = [c for c in preds.columns if c.endswith("-median")]
    assert len(candidates) == 1, f"Expected exactly one '-median' column, got {candidates}"
    return candidates[0]


def latest_metrics_csv(csv_logger) -> str:
    """Path to the metrics.csv actually written by this CSVLogger.

    `csv_logger.log_dir` is unreliable to read AFTER fit() - it's a lazily
    recomputed 'next available version' property, and can report a version
    one higher than what Trainer actually wrote (observed reproducibly with
    pytorch_lightning's CSVLogger here). Glob for the real thing instead.
    """
    version_dirs = sorted(
        Path(csv_logger.save_dir, csv_logger.name).glob("version_*"),
        key=lambda p: int(p.name.split("_")[1]),
    )
    if not version_dirs:
        raise FileNotFoundError(f"No version_* dir under {csv_logger.save_dir}/{csv_logger.name}")
    return str(version_dirs[-1] / "metrics.csv")


# ---- lag-52 seasonal-naive baseline + blend (the actual score lever) ----------------
# TFT smooths holiday amplitude (README SS4.6); lag-52 "same week last year" carries the
# full holiday peak (README SS4.1/4.3 - seasonality >> trend here). Blending the two
# recovers the amplitude TFT loses, on exactly the 5x-weighted holiday weeks. Matching is
# by week_of_year (not raw date arithmetic) so it's robust to holiday date-drift and
# 52/53-week years, and aligns "Thanksgiving week last year" -> "Thanksgiving week now".

def _week_of_year(date_series):
    """Raw ISO week number (1-53) from a date column. Computed fresh here rather than
    read off panel_df, whose own week_of_year column is StandardScaler-scaled."""
    return pd.to_datetime(date_series).dt.isocalendar().week.astype(int).values


def build_seasonal_lookup(hist_df, date_col="Date"):
    """Build the seasonal-naive lookups from a TRAIN-ONLY history frame (per fold, its
    own train portion - no leakage). hist_df needs Store, Dept, Weekly_Sales, and a date
    column. Returns a dict of the four fallback levels used by seasonal_series()."""
    h = hist_df[["Store", "Dept", date_col, "Weekly_Sales"]].copy()
    h["woy"] = _week_of_year(h[date_col])
    return {
        "store_dept_woy": h.groupby(["Store", "Dept", "woy"])["Weekly_Sales"].mean().to_dict(),
        "dept_woy": h.groupby(["Dept", "woy"])["Weekly_Sales"].mean().to_dict(),
        "store_woy": h.groupby(["Store", "woy"])["Weekly_Sales"].mean().to_dict(),
        "global": float(h["Weekly_Sales"].mean()),
    }


def seasonal_series(store_arr, dept_arr, woy_arr, lookup):
    """Vectorized seasonal-naive value per row, with fallback chain
    (Store,Dept,woy) -> (Dept,woy) -> (Store,woy) -> global mean."""
    sd, dp, st, gm = lookup["store_dept_woy"], lookup["dept_woy"], lookup["store_woy"], lookup["global"]
    out = np.empty(len(store_arr), dtype=float)
    for i, (s, d, w) in enumerate(zip(store_arr, dept_arr, woy_arr)):
        v = sd.get((s, d, w))
        if v is None:
            v = dp.get((d, w))
        if v is None:
            v = st.get((s, w))
        if v is None:
            v = gm
        out[i] = v
    return out


def apply_seasonal_blend(preds_df, pred_col, lookup, w, date_col="ds"):
    """Blend the TFT prediction with the seasonal-naive baseline:
    blended = w*TFT + (1-w)*seasonal, clipped to >= 0. preds_df needs Store, Dept, and a
    date column (week_of_year is derived from it, not read off a scaled column). Where
    seasonal is unavailable the fallback chain still returns a value (ultimately the
    global mean), so a blend is always defined. w=1.0 recovers pure (clipped) TFT;
    w=0.0 is pure seasonal-naive."""
    preds_df = preds_df.copy()
    woy = _week_of_year(preds_df[date_col])
    seasonal = seasonal_series(preds_df["Store"].values, preds_df["Dept"].values, woy, lookup)
    blended = w * preds_df[pred_col].values + (1.0 - w) * seasonal
    preds_df[pred_col] = np.clip(blended, 0.0, None)
    return preds_df


BLEND_WEIGHTS = [round(0.1 * i, 1) for i in range(11)]  # 0.0 .. 1.0 swept on the TESTLIKE fold


print("Columns fed as hist_exog_list (past covariates):", PAST_COVARIATE_COLS)
print("Columns fed as futr_exog_list (future covariates):", FUTURE_COVARIATE_COLS)
print("Columns fed as stat_exog_list (static covariates):", STATIC_COLS)


##  Build the long-format panel (gap-interpolated target, scaled covariates)

`neuralforecast` takes one long DataFrame (`unique_id`, `ds`, `y`, plus
exogenous columns) rather than Darts' per-series `TimeSeries` list. Target
gaps are linearly interpolated (interior missing weeks, not 0-sales weeks) -
same as the Darts notebooks. Covariates are built once per Store (they don't
vary by Dept) over the full train+test date range, ffill/bfill'd, then
merged onto every (Store, Dept) row - immune to any single department's
short test presence, same intent as the old `build_store_level_covariates`.

Numeric covariates (not the target) are scaled with a single `StandardScaler`
fit on **train dates only**, matching the old notebook's `df_scaler`.
**`IsHoliday` is deliberately excluded from scaling** - it's a 0/1 flag, not
a continuous quantity, and `weighted_mae` plus every holiday-weighted
diagnostic below matches against literal `True`/`False`; standardizing it
would turn holiday weeks into an arbitrary float (e.g. `-0.27`) that no
longer matches, silently zeroing out the competition's 5x holiday weighting
everywhere downstream. The target `y` is **not** manually scaled either -
`TFT(scaler_type="robust")` below normalizes/denormalizes each series
internally, keyed by `unique_id` identity, which is exactly the design flaw
that caused the Darts `Scaler` positional bug in the first place (see intro
markdown).

**Every series is reindexed through the same shared `TRAIN_END` date, not its
own last observed date.** `neuralforecast` forecasts the next `h` steps
immediately following each series' own last date in the fit data - if any
(Store, Dept) stopped selling before the shared train cutoff (a discontinued
department), its "last date" would be earlier than every other series', and
its true forecast window would silently land on the wrong calendar dates
entirely. Left as each series' own min/max range, this surfaces as
`nf.predict()` raising `"There are missing combinations of ids and times in
futr_df"` - not a smoke-test-only issue, it only shows up once real data has
a genuinely discontinued series. Any trailing gap this creates is linearly
interpolated same as interior gaps (holds the last known value flat when
there's no later real point to interpolate towards) - a reasonable "assume it
stayed flat" fallback for a handful of series, not a large distortion.


In [ ]:
def build_covariate_lookup(full_df, covariate_cols, global_start, global_end, freq=FREQ):
    """One row per (Store, Date) covariate snapshot, ffill/bfill'd per store,
    spanning the full global range so it always covers any forecast horizon."""
    store_level = full_df.drop_duplicates(subset=["Store", "Date"])[["Store", "Date"] + covariate_cols]
    full_range = pd.date_range(global_start, global_end, freq=freq)
    filled = []
    for store, g in store_level.groupby("Store"):
        g = g.sort_values("Date").set_index("Date").reindex(full_range)
        g.index.name = "Date"
        g[covariate_cols] = g[covariate_cols].ffill().bfill()
        g["Store"] = store
        filled.append(g.reset_index())
    lookup = pd.concat(filled, ignore_index=True)
    return lookup.rename(columns={"Date": "ds"})


def build_target_long(df, global_end, freq=FREQ):
    """Long (unique_id, ds, y, Store, Dept) frame, one row per (Store, Dept, Date).

    Every series is reindexed through the SAME global_end date (each series keeps
    its own natural start), not its own last observed date. This matters because
    neuralforecast forecasts the next `h` steps immediately following each
    series' own last date in the fit data - if a department stopped selling
    before the shared train cutoff, its "last date" would be earlier than every
    other series', and its true forecast window would land on the wrong
    calendar dates entirely (surfaces as `nf.predict()` raising "missing
    combinations of ids and times in futr_df"). Any trailing gap this creates
    is linearly interpolated same as interior gaps - `interpolate(limit_direction="both")`
    holds the last known value flat when there's no later real point to
    interpolate towards, which is a reasonable "assume it stayed flat" fallback
    for a handful of discontinued (Store, Dept) series, not a large distortion.
    """
    rows = []
    for (store, dept), g in df.groupby(["Store", "Dept"]):
        g = g.sort_values("Date")
        full_range = pd.date_range(g["Date"].min(), global_end, freq=freq)
        y = g.set_index("Date")["Weekly_Sales"].reindex(full_range)
        y = y.interpolate(method="linear", limit_direction="both")
        rows.append(pd.DataFrame({
            "unique_id": f"{store}_{dept}",
            "ds": full_range,
            "y": y.values,
            "Store": store,
            "Dept": dept,
        }))
    return pd.concat(rows, ignore_index=True)


numeric_cols = [c for c in all_covariate_cols if c != "IsHoliday"]  # IsHoliday stays a raw 0/1 flag - see markdown above
full_feat_df = pd.concat([train_feat, test_feat], ignore_index=True).drop_duplicates(
    subset=["Store", "Dept", "Date"]
)
df_scaler = StandardScaler()
train_dates = train_feat["Date"].unique()
df_scaler.fit(full_feat_df.loc[full_feat_df["Date"].isin(train_dates), numeric_cols])
full_feat_scaled_df = full_feat_df.copy()
full_feat_scaled_df[numeric_cols] = df_scaler.transform(full_feat_df[numeric_cols])

GLOBAL_START = full_feat_scaled_df["Date"].min()
GLOBAL_END = full_feat_scaled_df["Date"].max()
print(f"Covariates span {GLOBAL_START} -> {GLOBAL_END}")

covariate_lookup = build_covariate_lookup(full_feat_scaled_df, all_covariate_cols, GLOBAL_START, GLOBAL_END)

TRAIN_END = train_feat["Date"].max()
target_long = build_target_long(train_feat, global_end=TRAIN_END)

panel_df = target_long.merge(covariate_lookup, on=["Store", "ds"], how="left")
n_nan_cov = panel_df[all_covariate_cols].isna().sum().sum()
assert n_nan_cov == 0, f"panel_df: {n_nan_cov} NaN covariate values after merge - stop"

print(f"Built panel_df: {panel_df.shape[0]} rows, {panel_df['unique_id'].nunique()} (Store, Dept) series")
panel_df.head()


## Time-based CV: expanding-window splits

Uses the real shared `src/validation.py` splitter (`expanding_window_splits`)
- same one every other model in the project uses. **Deliberately cut to a
single fold** (`N_SPLITS = 1`): the user's real Colab run showed this 13-week
CV is misleadingly optimistic (CV WMAE ~1531 vs Kaggle 3566) because its
validation windows never contain Thanksgiving/Christmas, so it can neither
predict the real score nor tune the holiday-amplitude blend weight. The honest,
Kaggle-predictive number now comes from the **TESTLIKE 39-week holdout** below
(which does contain the holidays), and the freed compute (~28 min vs 3 folds)
goes to that plus the final fit - keeping the whole run comfortably under 2
hours. This 1-fold CV is kept only as a quick sanity check.


In [ ]:
from src.validation import expanding_window_splits, describe_split, Split

N_SPLITS = 1  # sanity check only - see markdown above; the honest number is TFT_TestLike
MIN_TRAIN_WEEKS = 52

splits = expanding_window_splits(
    train_feat["Date"], n_splits=N_SPLITS, val_weeks=VAL_WEEKS, min_train_weeks=MIN_TRAIN_WEEKS
)
assert len(splits) == N_SPLITS, "history too short for the requested number of folds"

for i, split in enumerate(splits):
    print(f"fold {i}:", describe_split(split))


##  Log covariate/static-covariate decisions (`TFT_Feature_Selection`)

Lightweight run - just records the covariate split, static-covariate scope,
and library choice as params. No model fit happens here (actual training
happens in the HP-search, CV, and Final stages below).


In [ ]:
with mlflow.start_run(run_name="TFT_Feature_Selection"):
    mlflow.log_params({
        "library": "neuralforecast",
        "future_covariates": ",".join(FUTURE_COVARIATE_COLS) if FUTURE_COVARIATE_COLS else "none",
        "past_covariates": ",".join(PAST_COVARIATE_COLS),
        "static_covariates": ",".join(STATIC_COLS),
        "static_covariate_scope": "Type (one-hot) + Size (scaled) - no Store/Dept identity embeddings",
        "likelihood": f"MQLoss({QUANTILES})",
        "scaling": "StandardScaler (covariates only, fit on train) + TFT(scaler_type='robust') for the target internally, per-series by unique_id",
        "target_gap_fill": "linear_interpolation (interior gaps only - genuine 0-sales weeks are untouched)",
        "covariate_gap_fill": "ffill/bfill, built once per Store (covariates are Dept-invariant)",
        "n_series_total": panel_df["unique_id"].nunique(),
        "input_size": INPUT_SIZE,
        "horizon_cv": VAL_WEEKS,
        "horizon_final": HORIZON,
        "start_padding_enabled": True,
    })

print("Logged TFT_Feature_Selection.")


##  Shared helper: build one CV fold's frames

Used by both the hyperparameter comparison below and the full CV loop, so
the exact same fold-construction logic backs both.


In [ ]:
def split_long(panel, split):
    """Apply a Split (src/validation.py) to the long panel_df. Returns (train_long, val_long)."""
    train_mask = panel["ds"] <= split.train_end
    if split.train_start is not None:
        train_mask &= panel["ds"] >= split.train_start
    val_mask = (panel["ds"] >= split.val_start) & (panel["ds"] <= split.val_end)
    return panel.loc[train_mask].copy(), panel.loc[val_mask].copy()


def fit_predict_fold(split, max_steps, max_time_minutes, batch_size, h=VAL_WEEKS,
                     hidden_size=128, n_head=4, dropout=0.1, logger=None):
    """Fit a fresh TFT on one fold's train frame, forecast its val window, and return
    (wmae, mae, scored_df, elapsed_seconds, nf). `wmae`/`scored` are for the RAW TFT
    median (no blend) - the blend weight is chosen separately on the TESTLIKE fold, and
    `scored` carries the raw predictions + actuals so a blend can be swept over it later.
    `h` lets the same helper serve both the 13-week CV folds and the 39-week TESTLIKE
    fold. Never reuses a model whose .fit() already ran - a fresh one is built each call."""
    fold_train, fold_val = split_long(panel_df, split)

    futr_df = fold_val[["unique_id", "ds"] + FUTURE_COVARIATE_COLS].copy()

    t0 = time.time()
    model = TFT(
        h=h,
        input_size=INPUT_SIZE,
        stat_exog_list=STATIC_COLS,
        hist_exog_list=PAST_COVARIATE_COLS,
        futr_exog_list=FUTURE_COVARIATE_COLS,
        loss=MQLoss(quantiles=QUANTILES),
        max_steps=max_steps,
        batch_size=batch_size,
        hidden_size=hidden_size,
        n_head=n_head,
        dropout=dropout,
        scaler_type="robust",
        random_seed=42,
        start_padding_enabled=True,
        **make_trainer_kwargs(logger=logger, max_time_minutes=max_time_minutes),
    )
    nf = NeuralForecast(models=[model], freq=FREQ)
    nf.fit(
        df=fold_train[["unique_id", "ds", "y"] + PAST_COVARIATE_COLS + FUTURE_COVARIATE_COLS],
        static_df=static_df,
    )
    preds = nf.predict(futr_df=futr_df)
    elapsed = time.time() - t0

    med_col = median_column(preds)
    scored = preds.merge(
        fold_val[["unique_id", "ds", "y", "Store", "Dept", "IsHoliday"]],
        on=["unique_id", "ds"], how="inner",
    )
    scored = scored.rename(columns={med_col: "Weekly_Sales_Pred", "y": "Weekly_Sales"})

    wmae = weighted_mae(scored["Weekly_Sales"].values, scored["Weekly_Sales_Pred"].values, scored["IsHoliday"].values)
    mae = float(np.mean(np.abs(scored["Weekly_Sales"].values - scored["Weekly_Sales_Pred"].values)))
    return wmae, mae, scored, elapsed, nf


##  Hyperparameter comparison (`TFT_HP_Baseline` / `TFT_HP_Wide` / `TFT_HP_Deep`)

A quick, tight-step-budget comparison of three architecture sizes on the most
recent CV fold (`splits[-1]`, the fold closest to the final fit's actual
history), to pick a configuration before committing to the full CV below.
Each candidate gets its own MLflow run so the comparison is inspectable
run-by-run in the DagsHub UI, not just as a printed table.


In [ ]:
HP_CONFIGS = [
    {"name": "Baseline", "hidden_size": 64, "n_head": 4, "dropout": 0.1},
    {"name": "Wide", "hidden_size": 128, "n_head": 4, "dropout": 0.1},
    {"name": "Deep", "hidden_size": 64, "n_head": 8, "dropout": 0.2},
]

hp_split = splits[-1]
hp_results = []

for cfg in HP_CONFIGS:
    cfg_kwargs = {k: v for k, v in cfg.items() if k != "name"}
    with mlflow.start_run(run_name=f"TFT_HP_{cfg['name']}"):
        mlflow.log_params({
            "library": "neuralforecast",
            "hp_candidate": cfg["name"],
            "max_steps": HP_MAX_STEPS,
            "batch_size": BATCH_SIZE_SEARCH,
            "fold": describe_split(hp_split),
            "input_size": INPUT_SIZE,
            "h": VAL_WEEKS,
            **cfg_kwargs,
        })

        wmae_hp, mae_hp, _, elapsed_hp, _ = fit_predict_fold(
            hp_split, max_steps=HP_MAX_STEPS, max_time_minutes=HP_MAX_TIME_MINUTES,
            batch_size=BATCH_SIZE_SEARCH, **cfg_kwargs
        )
        mlflow.log_metric("wmae", wmae_hp)
        mlflow.log_metric("mae", mae_hp)
        mlflow.log_metric("fit_predict_seconds", elapsed_hp)

    hp_results.append({"name": cfg["name"], "wmae": wmae_hp, "mae": mae_hp, "seconds": elapsed_hp, **cfg_kwargs})
    print(f"{cfg['name']}: WMAE={wmae_hp:.2f}  MAE={mae_hp:.2f}  ({elapsed_hp:.1f}s)")

hp_results_df = pd.DataFrame(hp_results).sort_values("wmae")
print(hp_results_df)

BEST_HP = hp_results_df.iloc[0].to_dict()
BEST_HP_KWARGS = {
    "hidden_size": int(BEST_HP["hidden_size"]),
    "n_head": int(BEST_HP["n_head"]),
    "dropout": float(BEST_HP["dropout"]),
}
print("Selected architecture for CV/Final:", BEST_HP_KWARGS)


##  Multi-fold expanding-window CV (`TFT_CV_Fold0` / `TFT_CV_Fold1` / `TFT_CV_Fold2`)

Refits a fresh TFT per fold with `BEST_HP_KWARGS` from the search above,
forecasts the fold's 13-week validation window, and scores with the
competition's `weighted_mae`. Each fold is logged as its own MLflow run
(rather than one run with three metrics) so all folds are individually
inspectable, and a per-fold training-loss curve (via `CSVLogger`) is logged
as an artifact on each. **Wall-clock time is printed and logged per fold** -
if a real run shows this is impractical within the available compute budget,
drop `N_SPLITS` to 1 above and re-run, documenting why (same spirit as the
old Darts notebook's own compute-driven scope reduction).


In [ ]:
fold_results = []
scored_folds = []

for i, split in enumerate(splits):
    csv_logger = CSVLogger(save_dir="../submissions/_tft_logs", name=f"cv_fold_{i}")

    with mlflow.start_run(run_name=f"TFT_CV_Fold{i}"):
        mlflow.log_params({
            "library": "neuralforecast",
            "cv_strategy": "expanding_window",
            "fold_index": i,
            "n_splits": N_SPLITS,
            "val_weeks": VAL_WEEKS,
            "min_train_weeks": MIN_TRAIN_WEEKS,
            "max_steps": CV_MAX_STEPS,
            "batch_size": BATCH_SIZE_SEARCH,
            "input_size": INPUT_SIZE,
            "future_covariates": ",".join(FUTURE_COVARIATE_COLS) if FUTURE_COVARIATE_COLS else "none",
            "past_covariates": ",".join(PAST_COVARIATE_COLS),
            **BEST_HP_KWARGS,
            **describe_split(split),
        })

        wmae_fold, mae_fold, scored_fold, elapsed_fold, _ = fit_predict_fold(
            split, max_steps=CV_MAX_STEPS, max_time_minutes=CV_MAX_TIME_MINUTES,
            batch_size=BATCH_SIZE_SEARCH, logger=csv_logger, **BEST_HP_KWARGS
        )
        scored_fold["fold"] = i

        mlflow.log_metric("wmae", wmae_fold)
        mlflow.log_metric("mae", mae_fold)
        mlflow.log_metric("fit_predict_seconds", elapsed_fold)

        try:
            loss_history = pd.read_csv(latest_metrics_csv(csv_logger))
            loss_col = "train_loss_epoch" if "train_loss_epoch" in loss_history else "train_loss"
            loss_plot_path = f"../submissions/_tft_train_loss_fold{i}.png"
            plt.figure(figsize=(6, 4))
            loss_curve = loss_history.dropna(subset=[loss_col]) if loss_col in loss_history else None
            if loss_curve is not None and not loss_curve.empty:
                plt.plot(loss_curve["step"], loss_curve[loss_col], marker="o")
            plt.title(f"TFT CV Fold {i} - training loss")
            plt.xlabel("Step")
            plt.ylabel("Train loss")
            plt.tight_layout()
            plt.savefig(loss_plot_path, dpi=120)
            plt.close()
            mlflow.log_artifact(loss_plot_path)
        except Exception as e:
            print(f"fold {i}: could not read/plot loss history ({e})")

    fold_results.append({**describe_split(split), "wmae": wmae_fold, "mae": mae_fold, "seconds": elapsed_fold})
    scored_folds.append(scored_fold)
    print(f"fold {i}: WMAE={wmae_fold:.2f}  MAE={mae_fold:.2f}  ({elapsed_fold:.1f}s)")

wmae_mean = float(np.mean([r["wmae"] for r in fold_results]))
wmae_std = float(np.std([r["wmae"] for r in fold_results]))
mae_mean = float(np.mean([r["mae"] for r in fold_results]))

scored = pd.concat(scored_folds, ignore_index=True)
print(f"\nCV WMAE: {wmae_mean:.2f} (+/- {wmae_std:.2f})")
print(f"CV MAE:  {mae_mean:.2f}")
print(f"Total CV wall-clock time: {sum(r['seconds'] for r in fold_results):.1f}s across {len(fold_results)} fold(s)")


##  TESTLIKE holdout + seasonal-blend weight sweep (`TFT_TestLike`)

**This is the honest, Kaggle-predictive validation, and where the blend weight
is chosen.** The 13-week CV above never sees a holiday, so its WMAE (~1500s)
badly under-predicts the real ~39-week Kaggle score - `README.md` documents the
same trap for Prophet, and fixed it with a "TESTLIKE" fold that copies the real
test structure. This fold does the same: train on everything up to **2011-10-28**,
then forecast the **39 weeks** `2011-11-04 -> 2012-07-27`, which contains
**Thanksgiving 2011 + Christmas 2011** - the exact holiday regime the real test
has, shifted one year earlier so its actuals are known.

TFT smooths those holiday peaks (README SS4.6); the **lag-52 seasonal-naive**
baseline (same-week-last-year mean, built here from the fold's own train slice
only) carries the full peak. We forecast once, then sweep the blend weight
`w in {0.0 .. 1.0}` over `w*TFT + (1-w)*seasonal` (clipped >= 0) by re-scoring -
no retraining per weight - and pick the `w` that minimizes `weighted_mae` on this
holiday-containing window. That `BEST_BLEND_W` is what the final submission uses.
`w=1.0` would mean pure TFT was already best; `w<1.0` means the seasonal baseline
genuinely helps on the 5x-weighted holiday weeks.


In [ ]:
# TESTLIKE window: mirror the real 39-week test one year earlier, holidays included.
date_grid = pd.DatetimeIndex(sorted(train_feat["Date"].unique()))


def _snap(target):
    return date_grid[date_grid.get_indexer([pd.Timestamp(target)], method="nearest")[0]]


tl_val_start = _snap("2011-11-04")
tl_val_end = _snap(pd.Timestamp(tl_val_start) + pd.Timedelta(weeks=HORIZON - 1))
tl_train_end = date_grid[date_grid < tl_val_start][-1]
testlike_split = Split(train_end=tl_train_end, val_start=tl_val_start, val_end=tl_val_end)
print("TESTLIKE split:", describe_split(testlike_split))

testlike_logger = CSVLogger(save_dir="../submissions/_tft_logs", name="testlike")
# h=HORIZON (39) here, unlike the 13-week CV folds - this fold IS the real horizon.
tl_wmae_raw, tl_mae_raw, tl_scored, tl_elapsed, _ = fit_predict_fold(
    testlike_split, max_steps=TESTLIKE_MAX_STEPS, max_time_minutes=TESTLIKE_MAX_TIME_MINUTES,
    batch_size=BATCH_SIZE_SEARCH, h=HORIZON, logger=testlike_logger, **BEST_HP_KWARGS
)
print(f"TESTLIKE raw-TFT WMAE (w=1.0): {tl_wmae_raw:.2f}  ({tl_elapsed:.1f}s)")

# Seasonal-naive lookup from the fold's OWN train slice only (<= tl_train_end) - no leakage.
tl_seasonal_lookup = build_seasonal_lookup(train_feat[train_feat["Date"] <= tl_train_end])

# Sweep the blend weight by re-scoring the same forecast - no retraining.
sweep_rows = []
for w in BLEND_WEIGHTS:
    blended = apply_seasonal_blend(tl_scored, "Weekly_Sales_Pred", tl_seasonal_lookup, w)
    wmae_w = weighted_mae(blended["Weekly_Sales"].values, blended["Weekly_Sales_Pred"].values, blended["IsHoliday"].values)
    sweep_rows.append({"w": w, "wmae": wmae_w})

sweep_df = pd.DataFrame(sweep_rows).sort_values("wmae").reset_index(drop=True)
print(sweep_df.to_string(index=False))

BEST_BLEND_W = float(sweep_df.iloc[0]["w"])
TESTLIKE_WMAE = float(sweep_df.iloc[0]["wmae"])
print(f"\nSelected blend weight BEST_BLEND_W = {BEST_BLEND_W} "
      f"(TESTLIKE WMAE {TESTLIKE_WMAE:.2f} vs raw-TFT {tl_wmae_raw:.2f})")

with mlflow.start_run(run_name="TFT_TestLike"):
    mlflow.log_params({
        "library": "neuralforecast",
        "fold": describe_split(testlike_split),
        "h": HORIZON,
        "max_steps": TESTLIKE_MAX_STEPS,
        "batch_size": BATCH_SIZE_SEARCH,
        "blend_baseline": "lag52_seasonal_naive_by_week_of_year",
        "blend_weights_swept": str(BLEND_WEIGHTS),
        **BEST_HP_KWARGS,
    })
    mlflow.log_metric("testlike_wmae_raw_tft", tl_wmae_raw)
    mlflow.log_metric("testlike_wmae_best_blend", TESTLIKE_WMAE)
    mlflow.log_metric("best_blend_weight", BEST_BLEND_W)
    for _, r in sweep_df.iterrows():
        mlflow.log_metric("testlike_wmae_by_w", float(r["wmae"]), step=int(round(r["w"] * 10)))

print("Logged TFT_TestLike.")


##  Diagnostics

Aggregate actual vs predicted, error distribution, holiday vs non-holiday
WMAE split, per-series WMAE, and WMAE broken down by store `Type` - a direct
readout of whether the static-covariate encoder is actually picking up
store-level differences. Pools predictions across all CV folds (`scored`
from the cell above).


In [ ]:
scored["error"] = scored["Weekly_Sales_Pred"] - scored["Weekly_Sales"]
scored["abs_error"] = scored["error"].abs()
scored["weight"] = scored["IsHoliday"].map({True: 5, False: 1})
scored = scored.merge(stores_df[["Store", "Type"]], on="Store", how="left")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

agg = scored.groupby("ds")[["Weekly_Sales", "Weekly_Sales_Pred"]].sum().sort_index()
axes[0, 0].plot(agg.index, agg["Weekly_Sales"], label="Actual", marker="o")
axes[0, 0].plot(agg.index, agg["Weekly_Sales_Pred"], label="Predicted", marker="o")
axes[0, 0].set_title("Aggregate Weekly Sales: Actual vs Predicted (Pooled CV Folds)")
axes[0, 0].set_xlabel("Date")
axes[0, 0].set_ylabel("Total Weekly Sales ($)")
axes[0, 0].legend()
axes[0, 0].tick_params(axis="x", rotation=45)

axes[0, 1].hist(scored["error"], bins=80, color="steelblue")
axes[0, 1].axvline(0, color="red", linestyle="--")
axes[0, 1].set_title("Prediction Error Distribution")
axes[0, 1].set_xlabel("Predicted - Actual")
axes[0, 1].set_ylabel("Count")

by_holiday = scored.groupby("IsHoliday")[["abs_error", "weight"]].apply(
    lambda g: (g["abs_error"] * g["weight"]).sum() / g["weight"].sum()
)
axes[1, 0].bar(["Non-Holiday", "Holiday"], by_holiday.values, color=["steelblue", "orange"])
axes[1, 0].set_title("WMAE: Holiday vs Non-Holiday Weeks")
axes[1, 0].set_ylabel("Weighted MAE")


def series_wmae(g):
    w = g["IsHoliday"].map({True: 5, False: 1})
    return (g["abs_error"] * w).sum() / w.sum()


per_series = scored.groupby(["Store", "Dept"])[["IsHoliday", "abs_error"]].apply(series_wmae).sort_values(ascending=False)
axes[1, 1].hist(per_series, bins=80, color="seagreen")
axes[1, 1].set_title("Per-Series WMAE Distribution")
axes[1, 1].set_xlabel("WMAE per (Store, Dept)")
axes[1, 1].set_ylabel("Count")

plt.tight_layout()
plt.savefig("../submissions/_tft_diagnostics.png", dpi=120)
plt.show()

by_type = scored.groupby("Type")[["abs_error", "weight"]].apply(
    lambda g: (g["abs_error"] * g["weight"]).sum() / g["weight"].sum()
).sort_index()

plt.figure(figsize=(6, 4))
plt.bar(by_type.index.astype(str), by_type.values, color=["#4c72b0", "#dd8452", "#55a868"])
plt.title("WMAE by Store Type (static covariate)")
plt.xlabel("Store Type")
plt.ylabel("Weighted MAE")
plt.tight_layout()
plt.savefig("../submissions/_tft_wmae_by_type.png", dpi=120)
plt.show()

print("Top 10 worst-performing series (highest WMAE):")
print(per_series.head(10))
print("\nWMAE by store Type:")
print(by_type)


##  Representative-series forecast plots with quantile bands

Best- and worst-WMAE series from the diagnostics above, each showing
history, actual, the median forecast, and the 10-90% quantile band from the
probabilistic TFT output.


In [ ]:
def plot_series_forecast(store, dept, split, ax, title_prefix):
    uid = f"{store}_{dept}"
    _, _, fold_scored, _, nf_fold = fit_predict_fold(
        split, max_steps=CV_MAX_STEPS, max_time_minutes=CV_MAX_TIME_MINUTES,
        batch_size=BATCH_SIZE_SEARCH, **BEST_HP_KWARGS
    )

    hist_df = panel_df.loc[(panel_df["unique_id"] == uid) & (panel_df["ds"] <= split.train_end)]
    series_scored = fold_scored.loc[fold_scored["unique_id"] == uid].sort_values("ds")

    lo_col = [c for c in series_scored.columns if c.endswith("-lo-80.0")]
    hi_col = [c for c in series_scored.columns if c.endswith("-hi-80.0")]

    ax.plot(hist_df["ds"], hist_df["y"], color="gray", label="History")
    ax.plot(series_scored["ds"], series_scored["Weekly_Sales"], color="black", marker="o", label="Actual")
    ax.plot(series_scored["ds"], series_scored["Weekly_Sales_Pred"], color="steelblue", marker="x", label="Median forecast")
    if lo_col and hi_col:
        ax.fill_between(
            series_scored["ds"], series_scored[lo_col[0]], series_scored[hi_col[0]],
            color="steelblue", alpha=0.2, label="10-90% band",
        )
    ax.axvline(split.train_end, color="red", linestyle="--", linewidth=1)
    ax.set_title(f"{title_prefix}: Store {store}, Dept {dept}")
    ax.legend(fontsize=8)
    ax.tick_params(axis="x", rotation=45)


best_key = per_series.index[-1]
worst_key = per_series.index[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_series_forecast(best_key[0], best_key[1], splits[-1], axes[0], "Best (lowest WMAE)")
plot_series_forecast(worst_key[0], worst_key[1], splits[-1], axes[1], "Worst (highest WMAE)")
plt.tight_layout()
plt.savefig("../submissions/_tft_representative_forecasts.png", dpi=120)
plt.show()


##  Required `.predict()` wrapper (`NeuralForecastPipeline`)

Every model needs a pipeline-shaped wrapper that takes raw, unpreprocessed
`test.csv`-shaped data and handles everything internally, so
`model_inference.ipynb` can call it directly. Contract is identical to the
old `DartsForecastPipeline`: raw test df in, `DataFrame[Id, Weekly_Sales]`
out. No per-series `Scaler`/`inverse_transform` bookkeeping needed here -
`neuralforecast` denormalizes each series internally by `unique_id` before
returning predictions.


In [ ]:
class NeuralForecastPipeline:
    def __init__(self, nf_model, preprocessor, df_scaler, numeric_cols,
                 future_covariate_cols, static_df, train_history_df,
                 seasonal_lookup, blend_weight, freq="W-FRI"):
        self.nf_model = nf_model
        self.preprocessor = preprocessor
        self.df_scaler = df_scaler
        self.numeric_cols = numeric_cols
        self.future_covariate_cols = future_covariate_cols
        self.static_df = static_df
        self.train_history_df = train_history_df
        # Seasonal-naive lookup (built from full train history) + the blend weight chosen
        # on the TESTLIKE fold, so the persisted pipeline reproduces the exact blended,
        # clipped forecast that was submitted - not the raw TFT median.
        self.seasonal_lookup = seasonal_lookup
        self.blend_weight = blend_weight
        self.freq = freq

    def predict(self, raw_test_df):
        test_clean = self.preprocessor.transform(raw_test_df)
        test_feat_local = add_temporal_features(test_clean)

        futr_local = test_feat_local.copy()
        futr_local["unique_id"] = futr_local["Store"].astype(str) + "_" + futr_local["Dept"].astype(str)
        futr_local["ds"] = futr_local["Date"]
        futr_local[self.numeric_cols] = self.df_scaler.transform(futr_local[self.numeric_cols])

        # neuralforecast expects a futr_df row for EVERY (unique_id, ds) it forecasts
        # for every series the model was fit on - not just whatever combos happen to
        # appear in raw test.csv. A (Store, Dept) that stopped selling and is genuinely
        # absent from test.csv (a real, known quirk of this dataset) would otherwise be
        # missing entirely from futr_df and crash nf.predict() with "missing combinations
        # of ids and times in futr_df". Build a full (fitted unique_id) x (test date)
        # grid instead, and fill it from a Store-level covariate lookup (covariates don't
        # vary by Dept, so any Store present in test_feat_local covers all its Depts,
        # fitted or not) - any resulting extra rows beyond what Kaggle needs are dropped
        # by the submission-reconciliation step below, same as always.
        test_dates = pd.to_datetime(sorted(test_feat_local["Date"].unique()))
        store_cov = build_covariate_lookup(
            futr_local, self.future_covariate_cols,
            test_dates.min(), test_dates.max(), freq=self.freq,
        )

        fitted_uids = self.static_df[["unique_id"]].copy()
        fitted_uids["Store"] = fitted_uids["unique_id"].str.split("_", n=1).str[0].astype(int)

        futr_df = (
            fitted_uids.merge(pd.DataFrame({"ds": test_dates}), how="cross")
            .merge(store_cov, on=["Store", "ds"], how="left")
        )
        missing = futr_df[self.future_covariate_cols].isna().any(axis=1).sum()
        assert missing == 0, f"{missing} futr_df rows missing future-covariate values after Store-level lookup"
        futr_df = futr_df[["unique_id", "ds"] + self.future_covariate_cols]

        preds = self.nf_model.predict(futr_df=futr_df)
        med_col = median_column(preds)
        preds = preds.rename(columns={med_col: "Weekly_Sales"})

        ids = preds["unique_id"].str.split("_", n=1, expand=True)
        preds["Store"] = ids[0].astype(int)
        preds["Dept"] = ids[1].astype(int)

        # Blend the raw TFT median with the lag-52 seasonal-naive baseline at the weight
        # chosen on the TESTLIKE fold, then clip >= 0 - this (not the raw median) is the
        # actual submission. apply_seasonal_blend derives week_of_year from "ds".
        preds = apply_seasonal_blend(preds, "Weekly_Sales", self.seasonal_lookup, self.blend_weight)

        preds["Id"] = (
            preds["Store"].astype(str) + "_" +
            preds["Dept"].astype(str) + "_" +
            preds["ds"].dt.strftime("%Y-%m-%d")
        )
        return preds[["Id", "Weekly_Sales"]]


##  Final fit on full history and log `TFT_Final`

Refits on all available train data with a fresh model instance, using
`BEST_HP_KWARGS` from the hyperparameter search and `h=HORIZON` (39, the
full test horizon - CV/HP folds only forecast 13 weeks). `start_padding_enabled=True`
means no series needs to be dropped for insufficient history, unlike the old
Darts notebook's `MIN_REQUIRED_LENGTH` filter. Wraps the fitted model, and
logs the wrapper (not the bare model) as the run artifact so
`model_inference.ipynb` can load it directly. A training-loss curve is
logged as an artifact on this run too.


In [ ]:
final_csv_logger = CSVLogger(save_dir="../submissions/_tft_logs", name="final")

final_model = TFT(
    h=HORIZON,
    input_size=INPUT_SIZE,
    stat_exog_list=STATIC_COLS,
    hist_exog_list=PAST_COVARIATE_COLS,
    futr_exog_list=FUTURE_COVARIATE_COLS,
    loss=MQLoss(quantiles=QUANTILES),
    max_steps=FINAL_MAX_STEPS,
    batch_size=BATCH_SIZE_FINAL,
    scaler_type="robust",
    random_seed=42,
    start_padding_enabled=True,
    **BEST_HP_KWARGS,
    **make_trainer_kwargs(logger=final_csv_logger, max_time_minutes=FINAL_MAX_TIME_MINUTES),
)
final_nf = NeuralForecast(models=[final_model], freq=FREQ)

t0 = time.time()
final_nf.fit(
    df=panel_df[["unique_id", "ds", "y"] + PAST_COVARIATE_COLS + FUTURE_COVARIATE_COLS],
    static_df=static_df,
)
final_fit_seconds = time.time() - t0
print(f"Final fit took {final_fit_seconds:.1f}s over {panel_df['unique_id'].nunique()} series")

# Seasonal-naive lookup from FULL train history (the final model trains on all of it),
# paired with BEST_BLEND_W chosen on the TESTLIKE fold - the pipeline applies this exact
# blend to the submission.
final_seasonal_lookup = build_seasonal_lookup(train_feat)

pipeline = NeuralForecastPipeline(
    nf_model=final_nf,
    preprocessor=preprocessor,
    df_scaler=df_scaler,
    numeric_cols=numeric_cols,
    future_covariate_cols=FUTURE_COVARIATE_COLS,
    static_df=static_df,
    train_history_df=train_feat,
    seasonal_lookup=final_seasonal_lookup,
    blend_weight=BEST_BLEND_W,
    freq=FREQ,
)

with mlflow.start_run(run_name="TFT_Final") as final_run:
    mlflow.log_params({
        "library": "neuralforecast",
        "max_steps": FINAL_MAX_STEPS,
        "batch_size": BATCH_SIZE_FINAL,
        "input_size": INPUT_SIZE,
        "horizon": HORIZON,
        "future_covariates": ",".join(FUTURE_COVARIATE_COLS) if FUTURE_COVARIATE_COLS else "none",
        "past_covariates": ",".join(PAST_COVARIATE_COLS),
        "static_covariates": ",".join(STATIC_COLS),
        "n_series_final": panel_df["unique_id"].nunique(),
        "cv_strategy": "expanding_window",
        "n_splits": N_SPLITS,
        "val_weeks": VAL_WEEKS,
        "submission": "TFT median blended with lag-52 seasonal-naive, clipped >= 0",
        "blend_weight": BEST_BLEND_W,
        "blend_weight_tuned_on": "TESTLIKE 39-week holdout (holidays included)",
        **BEST_HP_KWARGS,
    })
    mlflow.log_metric("cv_wmae_mean_at_selection", wmae_mean)
    mlflow.log_metric("cv_wmae_std_at_selection", wmae_std)
    mlflow.log_metric("cv_mae_mean_at_selection", mae_mean)
    mlflow.log_metric("testlike_wmae_best_blend", TESTLIKE_WMAE)
    mlflow.log_metric("best_blend_weight", BEST_BLEND_W)
    mlflow.log_metric("final_fit_seconds", final_fit_seconds)

    try:
        final_loss_history = pd.read_csv(latest_metrics_csv(final_csv_logger))
        loss_col = "train_loss_epoch" if "train_loss_epoch" in final_loss_history else "train_loss"
        loss_curve = final_loss_history.dropna(subset=[loss_col]) if loss_col in final_loss_history else None
        plt.figure(figsize=(6, 4))
        if loss_curve is not None and not loss_curve.empty:
            plt.plot(loss_curve["step"], loss_curve[loss_col], marker="o", color="darkorange")
        plt.title("TFT Final - training loss")
        plt.xlabel("Step")
        plt.ylabel("Train loss")
        plt.tight_layout()
        plt.savefig("../submissions/_tft_final_train_loss.png", dpi=120)
        plt.close()
        mlflow.log_artifact("../submissions/_tft_final_train_loss.png")
    except Exception as e:
        print(f"Could not read/plot final loss history ({e})")

    with tempfile.TemporaryDirectory() as tmp:
        pkl_path = f"{tmp}/tft_pipeline.pkl"
        with open(pkl_path, "wb") as f:
            pickle.dump(pipeline, f)
        mlflow.log_artifact(pkl_path, artifact_path="pipeline")

    final_run_id = final_run.info.run_id

print("Logged TFT_Final wrapper artifact, run_id:", final_run_id)


##  Generate submission CSV

Reconciles against `sampleSubmission.csv` exactly - drops any predicted rows
beyond what's required, fills any missing ones (series `neuralforecast` had
no history for at all) with a store/dept mean fallback, so the row count and
ID set match precisely. Also pushes the submission CSV and the diagnostics
PNGs to MLflow as artifacts on `TFT_Final`.


In [ ]:
submission = pipeline.predict(test_raw)

sample = pd.read_csv("../data/raw/sampleSubmission.csv")
required_ids = set(sample["Id"])

extra_ids = set(submission["Id"]) - required_ids
print("Extra rows generated beyond what's needed:", len(extra_ids))

submission_filtered = submission[submission["Id"].isin(required_ids)].copy()
missing_ids = required_ids - set(submission_filtered["Id"])
print("Still missing after filtering:", len(missing_ids))

fallback_lookup = train_feat.groupby(["Store", "Dept"])["Weekly_Sales"].mean().to_dict()
global_fallback = train_feat["Weekly_Sales"].mean()

missing_df = pd.DataFrame({"Id": sorted(missing_ids)})
parts = missing_df["Id"].str.rsplit("_", n=1, expand=True)
store_dept = parts[0].str.split("_", n=1, expand=True)
missing_df["Store"] = store_dept[0].astype(int)
missing_df["Dept"] = store_dept[1].astype(int)
missing_df["Weekly_Sales"] = missing_df.apply(
    lambda r: fallback_lookup.get((r["Store"], r["Dept"]), global_fallback), axis=1
)

final_submission = pd.concat(
    [submission_filtered, missing_df[["Id", "Weekly_Sales"]]], ignore_index=True
).sort_values("Id").reset_index(drop=True)

print("Final rows:", len(final_submission), "| Expected:", len(sample))
assert len(final_submission) == len(sample)
assert set(final_submission["Id"]) == required_ids

final_submission.to_csv("../submissions/tft_submission.csv", index=False)
final_submission.head()


In [ ]:
with mlflow.start_run(run_id=final_run_id):
    mlflow.log_artifact("../submissions/tft_submission.csv")
    mlflow.log_artifact("../submissions/_tft_diagnostics.png")
    mlflow.log_artifact("../submissions/_tft_wmae_by_type.png")
    mlflow.log_artifact("../submissions/_tft_representative_forecasts.png")

print("Submission + diagnostics logged as artifacts on TFT_Final (run_id:", final_run_id, ")")
